In [1]:
import pandas as pd

# Custom helper modules
import data_util
import plot_util

# CORGIS data import module
import electricity

In [2]:
utility = electricity.get_utility()
df = pd.json_normalize(utility)

ny_df = data_util.get_state_data('NY', df)
ny_df = data_util.prepare_data(ny_df)

# print(ny_df.columns)
# print(df.columns)

In [3]:
residential_customers_df = data_util.get_customer_utilities(
    ny_df, "Residential").round(2)

boxplot = plot_util.get_utility_type_box_plot(residential_customers_df)

In [4]:
key_metrics = ['SystemLossPercentage', 'LoadFactor', 'ResidentialUnitPrice',
               'IndustrialUnitPrice', 'FairnessIndex']
corr_matrix = ny_df[key_metrics].corr()

heatmap = plot_util.get_key_metrics_corr_matrix(corr_matrix)

In [5]:
# Keep residential utilities with non-zero system loss and residential load factor
residential_lf_sys_loss_df = data_util.get_residential_sys_loss(ny_df)
residential_lf_sys_loss_df = data_util.get_residential_load_factor(
    residential_lf_sys_loss_df).round(2)

dual_y_scatter = plot_util.get_fairness_dual_y_scatter_plot(
    residential_lf_sys_loss_df)

In [6]:
# Keep utilities that offer both industrial and residential rates
res_ind_customer_df = data_util.get_customer_utilities(ny_df, "Both").round(2)

dumbbell = plot_util.get_rate_disparity_dumbbell_plot(res_ind_customer_df)

In [7]:
# Look at a specific utility's energy usage/flow
utility_row = data_util.get_utility_usage(ny_df.iloc[10])
sankey = plot_util.get_energy_use_sankey_plot(utility_row)

# Look at the entire country's energy usage/flow

In [8]:
us_energy_flow = df.groupby(["Utility.State"]).sum().sum()[1:]
us_energy_flow = data_util.get_utility_usage(us_energy_flow, level="US")

us_sankey = plot_util.get_energy_use_sankey_plot(us_energy_flow)

In [9]:
# Gather all plots and export them as SVGs
plots = [boxplot, heatmap, dual_y_scatter, dumbbell, sankey, us_sankey]

plot_util.export_plots_as_svg(plots)